<a href="https://colab.research.google.com/github/Linda-Masia/MIT-805---Vincent-Mabuza-Linda-Masia/blob/Some-805/working_data_download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import requests

BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month:02d}.parquet"

def get_remote_file_size(url):
    """Get file size via HTTP HEAD request without downloading."""
    resp = requests.head(url, allow_redirects=True)
    return int(resp.headers.get("content-length", 0))

def select_files_for_target_size(years, target_gb=12):
    """Select monthly files cumulatively until target size (in GB) is reached."""
    target_bytes = target_gb * (1024 ** 3)
    selected = []
    running_total = 0

    for year in years:
        for month in range(1, 13):
            url = BASE_URL.format(year=year, month=month)
            try:
                size = get_remote_file_size(url)
            except Exception as e:
                print(f"Skipping {year}-{month:02d}: {e}")
                continue

            if size == 0:
                continue  # file doesn't exist for that month/year

            selected.append((year, month, url, size))
            running_total += size
            print(f"{year}-{month:02d}: {size / (1024**3):.2f} GB (running total: {running_total / (1024**3):.2f} GB)")

            if running_total >= target_bytes:
                return selected, running_total

    return selected, running_total

# Search across 2013-2023 until we hit ~12GB
selected_files, total_size = select_files_for_target_size(years=[2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023], target_gb=12)
print(f"\nSelected {len(selected_files)} files, total {total_size / (1024**3):.2f} GB")

2013-01: 0.17 GB (running total: 0.17 GB)
2013-02: 0.16 GB (running total: 0.33 GB)
2013-03: 0.18 GB (running total: 0.51 GB)
2013-04: 0.17 GB (running total: 0.68 GB)
2013-05: 0.18 GB (running total: 0.86 GB)
2013-06: 0.17 GB (running total: 1.03 GB)
2013-07: 0.16 GB (running total: 1.19 GB)
2013-08: 0.14 GB (running total: 1.33 GB)
2013-09: 0.16 GB (running total: 1.49 GB)
2013-10: 0.18 GB (running total: 1.67 GB)
2013-11: 0.17 GB (running total: 1.83 GB)
2013-12: 0.16 GB (running total: 2.00 GB)
2014-01: 0.16 GB (running total: 2.16 GB)
2014-02: 0.15 GB (running total: 2.31 GB)
2014-03: 0.18 GB (running total: 2.49 GB)
2014-04: 0.17 GB (running total: 2.66 GB)
2014-05: 0.17 GB (running total: 2.83 GB)
2014-06: 0.16 GB (running total: 2.99 GB)
2014-07: 0.15 GB (running total: 3.15 GB)
2014-08: 0.16 GB (running total: 3.31 GB)
2014-09: 0.17 GB (running total: 3.48 GB)
2014-10: 0.18 GB (running total: 3.66 GB)
2014-11: 0.17 GB (running total: 3.83 GB)
2014-12: 0.17 GB (running total: 4

In [ ]:
import os
import requests

DOWNLOAD_DIR = "data/working"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

for year, month, url, size in selected_files:
    filename = f"yellow_tripdata_{year}-{month:02d}.parquet"
    filepath = os.path.join(DOWNLOAD_DIR, filename)
    if not os.path.exists(filepath):
        print(f"Downloading {filename} ({size / (1024**3):.2f} GB)...")
        resp = requests.get(url, stream=True)
        with open(filepath, "wb") as f:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)
    else:
        print(f"Skipping {filename}: already exists.")